In [1]:
import os
import glob
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from itertools import product

In [3]:
import math

In [4]:
from scipy.signal import butter, filtfilt

In [5]:
import kneed
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler

# Open Sound Files

In [6]:
def load_config(directory):
    file_name=[f for f in glob.glob(directory + "/*.RNH")]
    f = open(file_name[0],'r')
    conf = f.read()    
    t = {}
    conf = conf.split("\n")
    for i in conf:
        if i != "":
            temp = i.split(",")
            t[temp[0]] = temp[1].strip()
    return t

In [7]:
def parse_date(date):
    return datetime.datetime.strptime(date,'%Y/%m/%d %H:%M:%S')

In [8]:
def data_prepare(file_dir):
    conf = load_config(file_dir)

    file_names = [f for f in glob.glob(file_dir + "/*.RND")]

    open_files = [open(f,'r').read().replace(' ',"").replace(",","").split('\n') for f in file_names]

    data_concat = [x for lst in open_files for x in lst]
    data_cleaned = ' '.join(data_concat).split()
    sound_lists = [float(data.replace('O','0')) for data in data_cleaned]

    start = parse_date(conf['Start Time'])
    time = [start + datetime.timedelta(milliseconds=(100*i)) for i in range(len(sound_lists))]
    df = pd.DataFrame({'Period start':time, 'Leq':sound_lists})

    return df

In [9]:
def cutoff(r, data):
    r = (100 - r) / 1000
    b, a = butter(4, r)
    filtered = filtfilt(b, a, data)
    return filtered

In [10]:
def standardize(data):
    sound = np.array(data)
    sound = sound.reshape(-1,1)

    scaler = StandardScaler()
    sound_normalized = scaler.fit_transform(sound)

    return sound_normalized

In [11]:
def sound_cluster(data):
    sse = {k : KMeans(n_clusters=k, random_state=0).fit(data).inertia_ for k in range(1, 5)}

    kn = kneed.KneeLocator(
        x=list(sse.keys()), 
        y=list(sse.values()), 
        curve='convex', 
        direction='decreasing', S=0.0)
    
    k_best = kn.knee
    kmeans_best = KMeans(n_clusters=k_best, random_state=0).fit(data)

    return kmeans_best

In [12]:
def find_peaks(data, time):
    data_copy = data.copy()
    X = time.reshape(-1, 1)
    dbscan = DBSCAN(eps=150, min_samples=10).fit(X)

    data_copy = data_copy.iloc[time]
    data_copy['peak_group'] = dbscan.labels_
    data_copy = data_copy[data_copy['peak_group'] != -1]
    return data_copy['peak_group']

In [13]:
def merge_data(df1, df2):
    df_merge = pd.concat([df1, df2], axis=1)
    df_merge['peak_group'] = df_merge['peak_group'].fillna(-1)
    return df_merge

In [14]:
def computeL_eq_t(data):
    sum_pow = np.sum(np.power(10, data/10))
    x = 1/len(data) * sum_pow
    return 10 * math.log(x,10)

def SoundAddition(data):
    x = np.sum(np.power(10, data/10))
    return 10 * math.log(x,10)

def computeSEL(data):
    return computeL_eq_t(data) + 10 * math.log((len(data)*100)/1000)

In [15]:
def IQR_outlier(data, d_type):
    Q1 = np.percentile(data[d_type], 25)
    Q3 = np.percentile(data[d_type], 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_result_cleaned = data[~((data[d_type] < lower_bound) | (data[d_type] > upper_bound))]
    return df_result_cleaned

In [16]:
one_hr = 36000

half_day = one_hr * 12
day = one_hr * 24
week = 7 * day

In [23]:
days_dict = {"half day":half_day, "day":day, "week":week}

In [18]:
def find_localpeak_alternative(d, data):
    df_copy = data.copy()
    df_copy = df_copy[:days_dict[d]+1]

    sound_cutoff = cutoff(70, df_copy['Leq'])
    df_copy = df_copy.assign(Leq_filtered=sound_cutoff)

    df_copy['Leq_norm'] = standardize(df_copy['Leq_filtered'])
    sound_norm = np.array(df_copy['Leq_norm']).reshape(-1,1)

    x_kmeans = sound_cluster(sound_norm)
    df_copy['cluster'] = x_kmeans.labels_
    
    avg_cluster = [df_copy[df_copy['cluster'] == i]['Leq_filtered'].mean() for i in range(len(np.unique(x_kmeans.labels_)))]
    max_cluster = np.array(avg_cluster).argmax()
    time_on_max_clusters = np.array(df_copy[df_copy['cluster'] == max_cluster].index)
    
    peaks = find_peaks(df_copy, time_on_max_clusters)
    df_result = merge_data(df_copy, peaks)

    peak_median = [np.median(df_result[df_result['peak_group'] == i].index) for i in peaks.unique() if (len(df_result[df_result['peak_group'] == i]) != 0)]
    unix_time = np.array(df_result['Period start'])
    results = []

    for c in peaks.unique():
        c_data = df_result[df_result['peak_group'] == c].index
        min_t, max_t = c_data.min(), c_data.max()
        peak_t = int(peak_median[c])

        in_which_hour = min_t//36000

        raw = np.array(df_result[df_result['peak_group'] == c]['Leq'])
        Leq = computeL_eq_t(raw)
        LAE = computeSEL(raw)

        r = {"hours":in_which_hour, "start_time": unix_time[min_t], "end_time": unix_time[max_t], "peak_time": peak_t,
             "interval": unix_time[max_t]-unix_time[min_t], "median":peak_median[c], "Leq":Leq, "Lae":LAE}

        results.append(r)

    df_result = pd.DataFrame(results)
    df_result['interval'] = df_result['interval'] / np.timedelta64(1, 's')

    df_result_cleaned = IQR_outlier(df_result, 'interval')

    return df_result_cleaned

In [19]:
def to_time(str_time):
	if (len(str_time) < 10) : return str_time
	else : return str_time[-9:]

def to_date(str_date):
	return str_date[:10]

def merge_time_date(str_time):
	if len(str_time.split(".")) > 1 :
		return datetime.datetime.strptime(str_time, '%Y-%m-%d %H:%M:%S.%f')
	else:
		return datetime.datetime.strptime(str_time, '%Y-%m-%d %H:%M:%S')

In [20]:
def find_accuracy(lag_width, flights_file, peaks):
    def match_cond(start,stop,date_time):
        return (date_time >= start-datetime.timedelta(milliseconds=lag_width*1000)) & (date_time <= stop+datetime.timedelta(milliseconds=lag_width*1000))
    
    flights_file["Local Time"] = flights_file["Local Time"].apply(lambda a: to_time(a))
    flights_file["Local Date"] = flights_file["Local Date"].apply(lambda a: to_date(a))
    
    flights_file["Datetime"] = flights_file["Local Date"] + " " + flights_file["Local Time"]
    flights_file["Datetime"] = flights_file["Datetime"].apply(lambda a: merge_time_date(a))
    
    peaks["start_time"] = peaks["start_time"].astype('str').apply(lambda a: merge_time_date(a))
    peaks["end_time"]   = peaks["end_time"].astype('str').apply(lambda a: merge_time_date(a))

    min_peak, max_peak = min(peaks["start_time"]), max(peaks["end_time"])
    
    flights_file = flights_file[(flights_file['Datetime'] > min_peak) & (flights_file['Datetime'] < max_peak)]
    
    matched = [flights_file[match_cond(start,stop,flights_file['Datetime'])] for start, stop in zip(peaks['start_time'], peaks['end_time'])
                if len(flights_file[match_cond(start,stop,flights_file['Datetime'])]) > 0]
    
    return len(matched) / len(peaks)

In [21]:
flights = pd.read_excel("../sound_proj_2/data/Raw-data/full_flights.xlsx", dtype={'Local Date': str, 'Local Time': str})

dir = {'Romruedee Romklao':"../sound_proj_2/data/Raw-data/1-Romruedee-Romklao-12-20-Mar-2014/AU1_0001/",
       'Central Village':"../sound_proj_2/data/Raw-data/2-Central-Village-3-11-Apr-2014/AU1_0002/",
       'Bangchalong Canal':"../sound_proj_2/data/Raw-data/3-Bangchalong-Canal-3-11-Apr-2014/AU1_0001/",
       'ThanaPlace':"../sound_proj_2/data/Raw-data/4-ThanaPlace-28-Apr-7-May-2014/AU1_0002/",
       'Romklao27':"../sound_proj_2/data/Raw-data/5-Romklao27-28-Apr-7-May-2014/AU1_0001/",
       'BangPla43':"../sound_proj_2/data/Raw-data/6-BangPla43-10-17-May-2014/AU1_0001/",
       'Bangsaothong':"../sound_proj_2/data/Raw-data/7-Bangsaothong-10-17-May-2014/AU1_0002/",
       'Habitia Ramindra':"../sound_proj_2/data/Raw-data/8-Habitia-Ramindra-20-26-May-2014/AU1_0001/"}

In [24]:
data_list = []

places = ['Romruedee Romklao', 'Central Village', 'Bangchalong Canal', 'ThanaPlace', 'Romklao27', 'BangPla43', 'Bangsaothong', 'Habitia Ramindra']
days = ['half day', 'day', 'week']
lag_width_list = [15,30,60]

for pl, d, lag_w in product(places, days, lag_width_list):
    df = data_prepare(dir[pl])
    df_sound = find_localpeak_alternative(d, df)
    acc = find_accuracy(lag_w, flights, df_sound)
    data_list.append({'place':pl, 'days':d, 'lag width': lag_w, 'accuracy score':acc})

In [25]:
df = pd.DataFrame(data_list)
df.to_csv("../sound_proj_2/data/Processed-data/output_viz.csv",index=False)

In [26]:
df

,place,days,lag width,accuracy score
0,Romruedee Romklao,half day,15,0.614035
1,Romruedee Romklao,half day,30,0.807018
2,Romruedee Romklao,half day,60,0.982456
3,Romruedee Romklao,day,15,0.640741
4,Romruedee Romklao,day,30,0.870370
...,...,...,...,...
67,Habitia Ramindra,day,30,0.613577
68,Habitia Ramindra,day,60,0.812010
69,Habitia Ramindra,week,15,0.447843
70,Habitia Ramindra,week,30,0.581649


# Data Viz

In [ ]:
import plotly.express as px

In [27]:
df_viz = pd.read_csv('../sound_proj_2/data/Processed-data/output_viz.csv')
df_viz.head()

,place,days,lag width,accuracy score
0,Romruedee Romklao,half day,15,0.614035
1,Romruedee Romklao,half day,30,0.807018
2,Romruedee Romklao,half day,60,0.982456
3,Romruedee Romklao,day,15,0.640741
4,Romruedee Romklao,day,30,0.870370


In [34]:
df_viz_half_day = df_viz[df_viz['days'] == 'half day']
df_viz_day = df_viz[df_viz['days'] == 'day']
df_viz_week = df_viz[df_viz['days'] == 'week']

In [41]:
fig = px.histogram(df_viz_half_day, x="place", y="accuracy score", color='lag width', text_auto='.4f'
                   , barmode='group', height=400, title="Accuracy score of Aircraft Calculate in half day")
fig.show()

In [42]:
fig = px.histogram(df_viz_day, x="place", y="accuracy score", color='lag width', text_auto='.4f'
                   , barmode='group', height=400, title="Accuracy score of Aircraft Calculate in day")
fig.show()

In [43]:
fig = px.histogram(df_viz_week, x="place", y="accuracy score", color='lag width', text_auto='.4f'
                   , barmode='group', height=400, title="Accuracy score of Aircraft Calculate in week")
fig.show()